# Joint 2-channel puncta segmentation (PRE + POST)

Fine-tune a single SwinUNETR (3-in / **2-out**) initialised from a
SimMIM+VICReg pretrained Swin-Tiny encoder, predicting two binary
puncta masks in one forward pass:

- channel 0 = PRE puncta  (boutons, render radius 4 px)
- channel 1 = POST puncta (PSDs,    render radius 2 px)

Supervision: Spotiflow pseudo-labels generated by
`scripts/pseudolabels/puncta_spotiflow_from_mip.py` (MIP mode).
Loss: per-channel Dice+BCE (`JointChannelDiceBCE`). The co-localised
"synapse" mask is post-processing on top of the trained heatmaps
(`colocalisation.build_synapse_mask`).

This notebook mirrors `train_swinunetr_pseudolabels.ipynb` end-to-end.


## Imports

In [ ]:
import os, sys, json, time
from datetime import datetime
from pathlib import Path


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from tqdm.auto import tqdm
import matplotlib.pyplot as plt


## Path setup

In [ ]:
NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / '.git').is_dir():
    REPO_ROOT = REPO_ROOT.parent
ROOT = REPO_ROOT / 'src'
for p in (ROOT, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print('REPO_ROOT:', REPO_ROOT)
print('ROOT     :', ROOT)


In [ ]:
# Training infrastructure
from synaptic_ssl.training.config import BaseCfg, DataCfg, ModelCfg, dump_config
from synaptic_ssl.training.seeding import seed_everything
from synaptic_ssl.training.logging import setup_logger, CSVMetricLogger
from synaptic_ssl.training.data import compute_channel_stats
from synaptic_ssl.training.lr_schedule import param_groups_layer_decay, make_warmup_cosine
from synaptic_ssl.training.checkpoints import save_checkpoint, load_checkpoint, find_latest_checkpoint

# Segmentation (joint 2-channel)
from synaptic_ssl.segmentation import (
    SegTrainCfg,
    JointChannelSegDataset,
    JointChannelDiceBCE, compute_dice_metric_per_channel, DiceBCELoss,
    build_swinunetr, load_pretrained_encoder_into_swinunetr, count_params,
    SegTrainTransform, SegValTransform,
    sliding_window_predict_multichannel, predict_full_image_multichannel,
    extract_pseudolabel_archive, discover_full_image_masks,
    SynapseColocCfg, build_synapse_mask,
)


## Configuration

In [ ]:
base_cfg = BaseCfg(
    seed             = 42,
    output_root      = '../../outputs',
    experiment_name  = 'swinunetr_joint_2ch',
    tag              = 'pretrained_enc',
    method_name      = 'swinunetr_seg_joint',
    init_source      = 'local_ckpt',
    # SSL-pretrained encoder (Swin-Tiny, SimMIM + VICReg)
    pretrained_ckpt_path = str(REPO_ROOT / 'data/checkpoints/pretrained_encoder_moby_fourier_vicreg_on.pt'),
    resume_path      = None,
    dry_run          = False,
)


In [ ]:
# Use all 8 acquisition sessions
SESSIONS = ['20251017', '20251030', '20251104', '20251205',
            '20251208', '20251213', '20251219', '20251220']

data_cfg = DataCfg(
    data_root        = str(REPO_ROOT / 'data/patches_128_from_zip'),
    exclude_patterns = ['KONTROLA'],
    val_split        = 0.15,
    batch_size       = 32,
    num_workers      = 4,
    pin_memory       = True,
    channel_names    = ['pre_synaptic', 'post_synaptic', 'structural'],
)


In [ ]:
model_cfg = ModelCfg(
    in_channels       = 3,
    img_size          = 128,
    feature_size      = 96,
    patch_size        = 2,
    window_size       = 7,
    depths            = (2, 2, 6, 2),
    num_heads         = (3, 6, 12, 24),
    dropout_path_rate = 0.1,
)


In [ ]:
# Joint-pipeline hyperparameters (plan §7)
seg_cfg = SegTrainCfg(
    epochs                = 12,
    warmup_epochs         = 1,
    base_lr               = 1e-4,
    decoder_lr            = 1e-3,
    weight_decay          = 0.05,
    layer_decay           = 0.85,
    grad_clip_norm        = 1.0,
    freeze_encoder_epochs = 1,
    save_every_n_epochs   = 2,
    val_metric_key        = 'dice_mean',
    val_metric_direction  = 'max',
    model_save_name       = 'swinunetr_joint_2ch_best.pt',
    dice_weight           = 1.0,
    bce_weight            = 1.0,
    dice_smooth           = 1.0,
    pseudolabel_cache_dir = None,  # unused; joint pipeline loads from a fixed mask dir
)


In [ ]:
# Pseudolabel source: a pre-extracted folder with one sub-dir per session.
# Extract with:  tar xzf data/pseudolabels/spotiflow.tar.gz -C data/pseudolabels/
PSEUDOLABEL_MASK_ROOT = REPO_ROOT / 'data/pseudolabels/spotiflow'

# Per-patch _pre/_post.npy dir (plan §3 format). Set if/when you regenerate
# pseudolabels with `scripts/pseudolabels/puncta_spotiflow_from_mip.py` in
# patches mode. Auto-fallback to the full-image mask root otherwise.
PER_PATCH_MASK_DIR = None


## Output directory and logger

In [ ]:
RUN_TS = datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir = Path(base_cfg.output_root) / f'{base_cfg.experiment_name}_{base_cfg.tag}_{RUN_TS}'
save_dir.mkdir(parents=True, exist_ok=True)
print('save_dir =', save_dir)


In [ ]:
logger = setup_logger('seg_joint', save_dir / 'run.log')
logger.info(f'experiment = {base_cfg.experiment_name}')
logger.info(f'tag        = {base_cfg.tag}')
logger.info(f'method     = {base_cfg.method_name}')
logger.info(f'init_source= {base_cfg.init_source}')
logger.info(f'save_dir   = {save_dir}')


In [ ]:
dump_config(
    save_dir / 'config.json',
    base=base_cfg, data=data_cfg, model=model_cfg, seg=seg_cfg,
    extra={'sessions': list(SESSIONS),
           'mask_root':          str(PSEUDOLABEL_MASK_ROOT),
           'per_patch_mask_dir': str(PER_PATCH_MASK_DIR) if PER_PATCH_MASK_DIR else None},
)
logger.info('config.json written')


## Seed and device

In [ ]:
generator = seed_everything(base_cfg.seed)
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f'seed   = {base_cfg.seed}')
logger.info(f'device = {device}')
if device.type == 'cuda':
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    logger.info(f'gpu    = {name}  ({mem:.1f} GB)')


## Pseudo-labels

The pseudo-labels are pre-extracted to `PSEUDOLABEL_MASK_ROOT` (one
full-reassembled `_pre.npy`/`_post.npy` per source image, organised as
`<mask_root>/<session>/<source_stem>_{pre,post}.npy`). The dataset
mmap-slices per-patch tiles via `grid_row`/`grid_col`/`patch_size` from
the patch `index.csv`. If the folder is missing, extract it once with::

    tar xzf data/pseudolabels/spotiflow.tar.gz -C data/pseudolabels/


In [ ]:
mask_root = Path(PSEUDOLABEL_MASK_ROOT)
assert mask_root.is_dir(), f'mask_root does not exist: {mask_root}'
logger.info(f'mask_root = {mask_root}')

mask_index = discover_full_image_masks(mask_root, sessions=SESSIONS)
logger.info(f'full-image mask sources indexed = {len(mask_index)}')
example_key = next(iter(mask_index))
logger.info(f'  example: {example_key} -> {mask_index[example_key]["pre"].name}')


## Data

In [ ]:
from synaptic_ssl.utils_data.patch_dataset import PatchDataset

raw_dataset = PatchDataset(
    root=data_cfg.data_root,
    exclude_patterns=data_cfg.exclude_patterns,
)
# Restrict to the configured sessions (nested layout: filename = '<session>/...')
session_set = set(SESSIONS)
raw_dataset.records = [r for r in raw_dataset.records
                       if Path(r['filename']).parts[0] in session_set]
logger.info(f'raw patches (all sessions) = {len(raw_dataset)}')
sample = raw_dataset[0]
logger.info(f'sample shape = {tuple(sample.shape)}  dtype = {sample.dtype}')
assert sample.ndim == 3 and sample.shape[0] == model_cfg.in_channels
assert sample.shape[-1] == model_cfg.img_size


In [ ]:
# Train/val split + per-channel stats (computed on train split)
n_val   = int(len(raw_dataset) * data_cfg.val_split)
n_train = len(raw_dataset) - n_val
train_subset, val_subset = random_split(raw_dataset, [n_train, n_val], generator=generator)
logger.info(f'split: train={n_train}  val={n_val}')

ch_mean, ch_std = compute_channel_stats(
    train_subset, in_channels=model_cfg.in_channels,
    max_samples=data_cfg.channel_stats_max_samples,
)
for name, m, s in zip(data_cfg.channel_names, ch_mean.tolist(), ch_std.tolist()):
    logger.info(f'  ch[{name:>14s}] mean={m:.5f}  std={s:.5f}')
stats = {'channel_names': list(data_cfg.channel_names),
         'mean': ch_mean.tolist(), 'std': ch_std.tolist()}
(save_dir / 'channel_stats.json').write_text(json.dumps(stats, indent=2))


In [ ]:
# Build index-filtered PatchDataset wrappers (matches blueprint pattern)
class IndexedPatchDataset:
    def __init__(self, base_ds, indices):
        self.root = base_ds.root
        self.channels = base_ds.channels
        self.records = [base_ds.records[i] for i in indices]
    def __len__(self):
        return len(self.records)

train_patch_ds = IndexedPatchDataset(raw_dataset, train_subset.indices)
val_patch_ds   = IndexedPatchDataset(raw_dataset, val_subset.indices)
logger.info(f'train patches = {len(train_patch_ds)}')
logger.info(f'val   patches = {len(val_patch_ds)}')


In [ ]:
# Joint 2-channel datasets
train_transform = SegTrainTransform(ch_mean, ch_std)
val_transform   = SegValTransform(ch_mean, ch_std)

train_seg_ds = JointChannelSegDataset(
    train_patch_ds,
    per_patch_mask_dir=PER_PATCH_MASK_DIR,
    full_image_mask_dir=mask_root,
    transform=train_transform,
)
val_seg_ds = JointChannelSegDataset(
    val_patch_ds,
    per_patch_mask_dir=PER_PATCH_MASK_DIR,
    full_image_mask_dir=mask_root,
    transform=val_transform,
)
logger.info(f'train seg dataset = {len(train_seg_ds)}')
logger.info(f'val   seg dataset = {len(val_seg_ds)}')


In [ ]:
# DataLoaders
common = dict(
    num_workers=data_cfg.num_workers,
    pin_memory=data_cfg.pin_memory,
    persistent_workers=data_cfg.num_workers > 0,
    prefetch_factor=2 if data_cfg.num_workers > 0 else None,
)
train_loader = DataLoader(train_seg_ds, batch_size=data_cfg.batch_size,
                          shuffle=True, drop_last=True, **common)
val_loader   = DataLoader(val_seg_ds,   batch_size=data_cfg.batch_size,
                          shuffle=False, **common)
logger.info(f'train batches = {len(train_loader)}  val batches = {len(val_loader)}')


## Sanity checks

In [ ]:
batch_img, batch_mask = next(iter(train_loader))
assert batch_img.shape == (data_cfg.batch_size, model_cfg.in_channels,
                           model_cfg.img_size, model_cfg.img_size)
assert batch_mask.shape == (data_cfg.batch_size, 2,
                            model_cfg.img_size, model_cfg.img_size)
assert batch_img.dtype == torch.float32
assert batch_mask.dtype == torch.float32
assert torch.isfinite(batch_img).all()
logger.info(f'[ok] batch shape: img={tuple(batch_img.shape)} mask={tuple(batch_mask.shape)}')
logger.info(f'     PRE coverage  = {batch_mask[:, 0].mean().item():.5f}')
logger.info(f'     POST coverage = {batch_mask[:, 1].mean().item():.5f}')


In [ ]:
# Quick visualisation: PRE+POST channels and the two mask channels
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
for i in range(min(4, data_cfg.batch_size)):
    pre_img  = batch_img[i, 0].numpy()
    post_img = batch_img[i, 1].numpy()
    pre_m    = batch_mask[i, 0].numpy()
    post_m   = batch_mask[i, 1].numpy()
    axes[0, i].imshow(pre_img,  cmap='gray'); axes[0, i].set_title(f'PRE input [{i}]');  axes[0, i].axis('off')
    axes[1, i].imshow(post_img, cmap='gray'); axes[1, i].set_title(f'POST input [{i}]'); axes[1, i].axis('off')
    axes[2, i].imshow(pre_m,    cmap='hot', vmin=0, vmax=1)
    axes[2, i].set_title(f'PRE mask  ({int(pre_m.sum())} px)');  axes[2, i].axis('off')
    axes[3, i].imshow(post_m,   cmap='hot', vmin=0, vmax=1)
    axes[3, i].set_title(f'POST mask ({int(post_m.sum())} px)'); axes[3, i].axis('off')
fig.suptitle('Joint 2-channel sanity batch', y=1.01)
fig.tight_layout()
fig.savefig(save_dir / 'sanity_batch.png', dpi=130, bbox_inches='tight')
plt.show()


## Model + SSL warm-start

In [ ]:
model = build_swinunetr(model_cfg, out_channels=2).to(device)
logger.info(f'SwinUNETR (out_channels=2) total params = {count_params(model) / 1e6:.2f} M')


In [ ]:
enc_summary = load_pretrained_encoder_into_swinunetr(
    model, base_cfg.pretrained_ckpt_path, logger=logger,
)
n_loaded = enc_summary.get('n_loaded', 0)
n_target = enc_summary.get('n_target_params', 0) or 1
frac = n_loaded / n_target
logger.info(f'encoder load: {n_loaded}/{n_target}  ({frac:.1%})')
if frac < 0.95:
    logger.warning('encoder load fraction below 95%% -- check ModelCfg matches SSL pretrain')
if enc_summary.get('channel_mean') is not None:
    logger.info(f'  pretrained channel_mean = {enc_summary["channel_mean"]}')
    logger.info(f'  pretrained channel_std  = {enc_summary["channel_std"]}')


In [ ]:
model.eval()
with torch.no_grad():
    test_out = model(batch_img[:2].to(device))
assert test_out.shape == (2, 2, model_cfg.img_size, model_cfg.img_size), f'got {test_out.shape}'
logger.info(f'[ok] forward pass: input {tuple(batch_img[:2].shape)} -> output {tuple(test_out.shape)}')
model.train()


In [ ]:
# Gradient flow sanity
loss_fn = JointChannelDiceBCE(
    dice_weight=seg_cfg.dice_weight,
    bce_weight=seg_cfg.bce_weight,
    smooth=seg_cfg.dice_smooth,
)
_opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
_opt.zero_grad(set_to_none=True)
_img = batch_img[:2].to(device)
_msk = batch_mask[:2].to(device)
_out = model(_img)
_losses = loss_fn(_out, _msk)
_losses['loss'].backward()
n_grad = sum(1 for p in model.parameters() if p.grad is not None)
n_tot  = sum(1 for _ in model.parameters())
_opt.step()
del _opt, _losses, _out, _img, _msk
logger.info(f'[ok] grad flow: {n_grad}/{n_tot} params got gradients')


## Overfit check

Drive loss to ~0 on a single mini-batch. PRE+POST per-channel Dice should
both reach > 0.95 within ~100 steps.


In [ ]:
RUN_OVERFIT     = True
N_OVERFIT_STEPS = 100
OVERFIT_LR      = 5e-4


In [ ]:
if RUN_OVERFIT:
    of_model = build_swinunetr(model_cfg, out_channels=2).to(device)
    of_opt = torch.optim.AdamW(of_model.parameters(), lr=OVERFIT_LR)
    of_img = batch_img.to(device)
    of_msk = batch_mask.to(device)

    of_loss_hist, of_dice_pre_hist, of_dice_post_hist = [], [], []
    of_model.train()
    for step in tqdm(range(N_OVERFIT_STEPS), desc='overfit'):
        of_opt.zero_grad(set_to_none=True)
        out = of_model(of_img)
        losses = loss_fn(out, of_msk)
        losses['loss'].backward()
        of_opt.step()
        of_loss_hist.append(losses['loss'].item())
        with torch.no_grad():
            d = compute_dice_metric_per_channel(out, of_msk)
            of_dice_pre_hist.append(float(d[0]))
            of_dice_post_hist.append(float(d[1]))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(of_loss_hist); ax1.set_xlabel('step'); ax1.set_ylabel('loss')
    ax1.set_title('Overfit loss'); ax1.grid(True, alpha=0.3)
    ax2.plot(of_dice_pre_hist,  label='PRE',  color='tab:red')
    ax2.plot(of_dice_post_hist, label='POST', color='tab:blue')
    ax2.set_xlabel('step'); ax2.set_ylabel('Dice'); ax2.set_title('Overfit Dice')
    ax2.legend(); ax2.grid(True, alpha=0.3)
    fig.suptitle(f'Overfit check: {N_OVERFIT_STEPS} steps on 1 batch')
    fig.tight_layout()
    fig.savefig(save_dir / 'overfit_check.png', dpi=150, bbox_inches='tight')
    plt.show()
    logger.info(f'overfit: loss={of_loss_hist[-1]:.5f}  '
                f'dice_pre={of_dice_pre_hist[-1]:.4f}  '
                f'dice_post={of_dice_post_hist[-1]:.4f}')

    del of_model, of_opt, of_img, of_msk
    if device.type == 'cuda':
        torch.cuda.empty_cache()
else:
    logger.info('overfit check skipped')


## Full training

### Reset model + load SSL encoder

In [ ]:
generator = seed_everything(base_cfg.seed)

model = build_swinunetr(model_cfg, out_channels=2).to(device)
if base_cfg.pretrained_ckpt_path is not None:
    load_pretrained_encoder_into_swinunetr(
        model, base_cfg.pretrained_ckpt_path, logger=logger,
    )
logger.info(f'[reset] model params = {count_params(model) / 1e6:.2f} M')


### Optimiser + scheduler + AMP

In [ ]:
# LLRD on encoder, flat LR on decoder
encoder_groups = param_groups_layer_decay(
    model.swinViT,
    base_lr=seg_cfg.base_lr,
    weight_decay=seg_cfg.weight_decay,
    layer_decay=seg_cfg.layer_decay,
)
encoder_param_ids = {id(p) for p in model.swinViT.parameters()}
decoder_params = [p for p in model.parameters() if id(p) not in encoder_param_ids]
decoder_group = {'params': decoder_params,
                 'lr': seg_cfg.decoder_lr,
                 'weight_decay': seg_cfg.weight_decay}
optimizer = torch.optim.AdamW(encoder_groups + [decoder_group], betas=(0.9, 0.999))
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, make_warmup_cosine(seg_cfg.warmup_epochs, seg_cfg.epochs),
)

# bf16 on A100/H100, fp16 elsewhere. Detect via compute capability.
use_amp = device.type == 'cuda'
if use_amp:
    cap = torch.cuda.get_device_capability(0)
    if cap[0] >= 8:                            # Ampere or newer -> bf16
        amp_dtype, use_fp16 = torch.bfloat16, False
    else:
        amp_dtype, use_fp16 = torch.float16, True
else:
    amp_dtype, use_fp16 = torch.float32, False
scaler = torch.amp.GradScaler(device.type, enabled=use_fp16)
logger.info(f'AMP: enabled={use_amp}  dtype={amp_dtype}  fp16_scaler={use_fp16}')

logger.info(f'encoder param groups = {len(encoder_groups)}')
logger.info(f'decoder params       = {sum(p.numel() for p in decoder_params) / 1e6:.2f} M')


### Resume from checkpoint (optional)

In [ ]:
start_epoch     = 1
best_val_metric = float('-inf')
best_epoch      = 0

_resume = base_cfg.resume_path
if _resume is not None:
    p = Path(_resume)
    if p.is_dir():
        p = find_latest_checkpoint(p)
    if p is None or not Path(p).exists():
        logger.warning(f'resume path {_resume!r} not found -- starting fresh')
    else:
        ckpt = load_checkpoint(
            p, encoder=model,
            optimizer=optimizer, scheduler=scheduler, scaler=scaler,
            map_location=device,
        )
        start_epoch     = int(ckpt.get('epoch', 0)) + 1
        best_val_metric = ckpt.get('val_metric', best_val_metric) or best_val_metric
        best_epoch      = int(ckpt.get('epoch', 0))
        logger.info(f'resumed from {p} (epoch {ckpt.get("epoch")})')
logger.info(f'start_epoch = {start_epoch}  best_val_metric = {best_val_metric}')


### CSV metric logger

In [ ]:
csv_fields = [
    'epoch', 'phase',
    'train_loss', 'train_dice_loss', 'train_bce_loss',
    'train_pre_loss', 'train_post_loss',
    'val_loss', 'val_dice_pre', 'val_dice_post', 'val_dice_mean',
    'lr_encoder', 'lr_decoder',
    'epoch_time_s', 'train_time_s', 'val_time_s',
    'best_val_metric', 'best_epoch',
    'grad_norm_mean', 'grad_norm_max',
]
csv_logger = CSVMetricLogger(save_dir / 'metrics.csv', csv_fields)


### Training-loop helpers

In [ ]:
def freeze_encoder(mdl, freeze):
    for p in mdl.swinViT.parameters():
        p.requires_grad = not freeze

def get_lrs(opt):
    enc_lrs = [g['lr'] for g in opt.param_groups if g.get('stage') is not None]
    dec_lrs = [g['lr'] for g in opt.param_groups if g.get('stage') is None]
    return (max(enc_lrs) if enc_lrs else 0.0,
            dec_lrs[0] if dec_lrs else 0.0)

def all_trainable_params(mdl):
    return [p for p in mdl.parameters() if p.requires_grad]


### Qualitative monitoring grid (plan §4.5)

In [ ]:
# Cache a fixed 16-patch grid of val indices so checkpoints can be compared.
N_QUAL = 16
rng = np.random.default_rng(base_cfg.seed)
qual_idx = rng.choice(len(val_seg_ds), size=min(N_QUAL, len(val_seg_ds)), replace=False)
qual_idx = sorted(int(i) for i in qual_idx)
logger.info(f'qualitative grid indices = {qual_idx[:8]}... ({len(qual_idx)} total)')

(save_dir / 'qualitative').mkdir(exist_ok=True)

def save_qualitative_grid(epoch: int):
    model.eval()
    rows = []
    with torch.no_grad():
        for i in qual_idx:
            img, mask = val_seg_ds[i]
            img_t = img.unsqueeze(0).to(device)
            with torch.amp.autocast(device.type, enabled=use_amp, dtype=amp_dtype):
                logits = model(img_t)
            probs = torch.sigmoid(logits)[0].cpu().numpy()           # (2, H, W)
            rows.append((img.numpy(), mask.numpy(), probs))
    n = len(rows)
    fig, axes = plt.subplots(n, 6, figsize=(18, 3 * n))
    if n == 1:
        axes = axes[None, :]
    col_titles = ['PRE input', 'POST input',
                  'pred PRE', 'pred POST',
                  'pseudo PRE', 'pseudo POST']
    for r, (img_np, mask_np, prob_np) in enumerate(rows):
        # input channels are channel-normalised; show as-is with percentile stretch
        for c, (arr, cmap, title) in enumerate([
            (img_np[0], 'gray', col_titles[0]),
            (img_np[1], 'gray', col_titles[1]),
            (prob_np[0], 'hot', col_titles[2]),
            (prob_np[1], 'hot', col_titles[3]),
            (mask_np[0], 'hot', col_titles[4]),
            (mask_np[1], 'hot', col_titles[5]),
        ]):
            ax = axes[r, c]
            if c < 2:
                lo, hi = np.percentile(arr, (1, 99))
                ax.imshow(arr, cmap=cmap, vmin=lo, vmax=max(hi, lo + 1e-3))
            else:
                ax.imshow(arr, cmap=cmap, vmin=0, vmax=1)
            if r == 0:
                ax.set_title(title)
            ax.axis('off')
    fig.suptitle(f'Qualitative grid @ epoch {epoch}', y=1.001)
    fig.tight_layout()
    fig.savefig(save_dir / 'qualitative' / f'epoch_{epoch:03d}.png',
                dpi=120, bbox_inches='tight')
    plt.close(fig)
    model.train()


### Full training loop

In [ ]:
train_losses, val_dice_means, lr_history = [], [], []
total_t0 = time.time()


In [ ]:
_better = lambda new, best: new > best   # mean Dice: higher is better

if base_cfg.dry_run:
    logger.info('dry_run=True -- skipping full training loop')
else:
    for epoch in range(start_epoch, seg_cfg.epochs + 1):
        in_warmup = epoch <= seg_cfg.freeze_encoder_epochs
        freeze_encoder(model, in_warmup)
        phase = 'frozen' if in_warmup else 'full'

        # ---- TRAIN ----
        model.train()
        running = {'loss': 0.0, 'dice_loss': 0.0, 'bce_loss': 0.0,
                   'pre_loss': 0.0, 'post_loss': 0.0}
        grads = []
        t_train = time.time()
        pbar = tqdm(train_loader, desc=f'ep {epoch}/{seg_cfg.epochs} [{phase}]', leave=False)
        for img_b, mask_b in pbar:
            img_b  = img_b.to(device, non_blocking=True)
            mask_b = mask_b.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device.type, enabled=use_amp, dtype=amp_dtype):
                logits = model(img_b)
                losses = loss_fn(logits, mask_b)
            if use_fp16:
                scaler.scale(losses['loss']).backward()
                scaler.unscale_(optimizer)
                gn = torch.nn.utils.clip_grad_norm_(
                    all_trainable_params(model), max_norm=seg_cfg.grad_clip_norm,
                )
                scaler.step(optimizer); scaler.update()
            else:
                losses['loss'].backward()
                gn = torch.nn.utils.clip_grad_norm_(
                    all_trainable_params(model), max_norm=seg_cfg.grad_clip_norm,
                )
                optimizer.step()
            grads.append(gn.item())
            for k in running:
                running[k] += losses[k].item()
            pbar.set_postfix(loss=f'{losses["loss"].item():.4f}')
        n_tb = max(1, len(train_loader))
        train_loss      = running['loss']      / n_tb
        train_dice_loss = running['dice_loss'] / n_tb
        train_bce_loss  = running['bce_loss']  / n_tb
        train_pre_loss  = running['pre_loss']  / n_tb
        train_post_loss = running['post_loss'] / n_tb
        train_time = time.time() - t_train
        scheduler.step()

        # ---- VALIDATE ----
        model.eval()
        val_loss_sum, val_dpre_sum, val_dpost_sum, n_vb = 0.0, 0.0, 0.0, 0
        t_val = time.time()
        with torch.no_grad():
            for img_b, mask_b in val_loader:
                img_b  = img_b.to(device, non_blocking=True)
                mask_b = mask_b.to(device, non_blocking=True)
                with torch.amp.autocast(device.type, enabled=use_amp, dtype=amp_dtype):
                    logits = model(img_b)
                    v_losses = loss_fn(logits, mask_b)
                val_loss_sum += v_losses['loss'].item()
                d = compute_dice_metric_per_channel(logits, mask_b)
                val_dpre_sum  += float(d[0])
                val_dpost_sum += float(d[1])
                n_vb += 1
        val_loss      = val_loss_sum / max(1, n_vb)
        val_dice_pre  = val_dpre_sum / max(1, n_vb)
        val_dice_post = val_dpost_sum / max(1, n_vb)
        val_dice_mean = 0.5 * (val_dice_pre + val_dice_post)
        val_time = time.time() - t_val
        val_metric = val_dice_mean

        enc_lr, dec_lr = get_lrs(optimizer)
        epoch_time = train_time + val_time
        gmean = sum(grads) / max(1, len(grads))
        gmax  = max(grads) if grads else 0.0

        # ---- CHECKPOINTING ----
        improved = ''
        if _better(val_metric, best_val_metric):
            best_val_metric = val_metric
            best_epoch      = epoch
            improved        = ' *best*'
            save_checkpoint(
                save_dir / 'best_model.pt',
                encoder=model, heads=None,
                optimizer=optimizer, scheduler=scheduler, scaler=scaler,
                epoch=epoch, val_metric=val_metric, train_loss=train_loss,
                extra={'channel_mean': ch_mean.tolist(),
                       'channel_std':  ch_std.tolist(),
                       'val_dice_pre':  val_dice_pre,
                       'val_dice_post': val_dice_post,
                       'val_dice_mean': val_dice_mean},
            )
        save_checkpoint(
            save_dir / 'last.pt',
            encoder=model, heads=None,
            optimizer=optimizer, scheduler=scheduler, scaler=scaler,
            epoch=epoch, val_metric=val_metric, train_loss=train_loss,
            extra={'channel_mean': ch_mean.tolist(),
                   'channel_std':  ch_std.tolist()},
        )
        if seg_cfg.save_every_n_epochs and epoch % seg_cfg.save_every_n_epochs == 0:
            save_checkpoint(
                save_dir / f'epoch_{epoch:04d}.pt',
                encoder=model, heads=None,
                optimizer=optimizer, scheduler=scheduler, scaler=scaler,
                epoch=epoch, val_metric=val_metric, train_loss=train_loss,
            )
            try:
                save_qualitative_grid(epoch)
            except Exception as e:
                logger.warning(f'qualitative grid failed at epoch {epoch}: {e}')

        # ---- LOGGING ----
        train_losses.append(train_loss)
        val_dice_means.append(val_dice_mean)
        lr_history.append((enc_lr, dec_lr))
        csv_logger.log(dict(
            epoch=epoch, phase=phase,
            train_loss=train_loss, train_dice_loss=train_dice_loss, train_bce_loss=train_bce_loss,
            train_pre_loss=train_pre_loss, train_post_loss=train_post_loss,
            val_loss=val_loss,
            val_dice_pre=val_dice_pre, val_dice_post=val_dice_post, val_dice_mean=val_dice_mean,
            lr_encoder=enc_lr, lr_decoder=dec_lr,
            epoch_time_s=epoch_time, train_time_s=train_time, val_time_s=val_time,
            best_val_metric=best_val_metric, best_epoch=best_epoch,
            grad_norm_mean=gmean, grad_norm_max=gmax,
        ))
        logger.info(
            f'ep {epoch:3d}/{seg_cfg.epochs} [{phase}]  '
            f'train={train_loss:.5f} (pre={train_pre_loss:.4f} post={train_post_loss:.4f})  '
            f'val_loss={val_loss:.4f}  '
            f'dice(pre/post/mean)={val_dice_pre:.3f}/{val_dice_post:.3f}/{val_dice_mean:.3f}  '
            f'lr(enc/dec)={enc_lr:.2e}/{dec_lr:.2e}  t={epoch_time:.1f}s  gn={gmean:.2f}{improved}'
        )
    csv_logger.close()
    logger.info(f'total training time = {(time.time() - total_t0) / 60:.1f} min')


## After training

In [ ]:
# Reload best checkpoint
best_path = save_dir / 'best_model.pt'
if best_path.exists():
    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    if 'encoder_state_dict' in ckpt:
        model.load_state_dict(ckpt['encoder_state_dict'])
    logger.info(
        f'reloaded best model from epoch {ckpt.get("epoch")}  '
        f'dice_mean={ckpt.get("val_dice_mean", ckpt.get("val_metric")):.4f}  '
        f'dice_pre={ckpt.get("val_dice_pre", float("nan")):.4f}  '
        f'dice_post={ckpt.get("val_dice_post", float("nan")):.4f}'
    )
else:
    logger.warning('no best_model.pt found')


In [ ]:
# Quick training-curve plot
import csv as _csv
rows = list(_csv.DictReader(open(save_dir / 'metrics.csv')))
ep = [int(r['epoch']) for r in rows]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(ep, [float(r['train_loss']) for r in rows], label='train total')
ax1.plot(ep, [float(r['train_pre_loss'])  for r in rows], label='train PRE',  ls='--')
ax1.plot(ep, [float(r['train_post_loss']) for r in rows], label='train POST', ls='--')
ax1.plot(ep, [float(r['val_loss']) for r in rows], label='val total')
ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax1.set_title('Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(ep, [float(r['val_dice_pre'])  for r in rows], label='val Dice PRE')
ax2.plot(ep, [float(r['val_dice_post']) for r in rows], label='val Dice POST')
ax2.plot(ep, [float(r['val_dice_mean']) for r in rows], label='val Dice mean', lw=2)
ax2.set_xlabel('epoch'); ax2.set_ylabel('Dice'); ax2.set_title('Dice (vs pseudolabels)')
ax2.legend(); ax2.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(save_dir / 'training_curves.png', dpi=140, bbox_inches='tight')
plt.show()


## Full-image sliding-window inference (multi-channel)

In [ ]:
from synaptic_ssl.utils_data.reassemble import reassemble_image, list_image_indices

available = list_image_indices(data_cfg.data_root, exclude_patterns=data_cfg.exclude_patterns)
logger.info(f'available full images: {len(available)}')

demo_idx = available[0]
logger.info(f'demo image index = {demo_idx}')
full_image, full_records = reassemble_image(
    data_cfg.data_root, demo_idx, exclude_patterns=data_cfg.exclude_patterns,
)
logger.info(f'full image shape = {full_image.shape}')


In [ ]:
model.eval()
prob_2ch = sliding_window_predict_multichannel(
    model, full_image,
    patch_size=model_cfg.img_size,
    ch_mean=ch_mean, ch_std=ch_std,
    device=device, overlap=0.5, batch_size=16,
    n_out_channels=2,
)
logger.info(f'prob_2ch shape = {prob_2ch.shape}  PRE [{prob_2ch[0].min():.3f}, {prob_2ch[0].max():.3f}]  '
            f'POST [{prob_2ch[1].min():.3f}, {prob_2ch[1].max():.3f}]')

mask_2ch  = (prob_2ch >= 0.5).astype(np.uint8)
mask_pre, mask_post = mask_2ch[0], mask_2ch[1]


In [ ]:
# Overlay: input PRE / POST, predicted PRE / POST, pseudolabel PRE / POST
session = Path(full_records[0]['filename']).parts[0] \
          if 'filename' in full_records[0] else SESSIONS[0]
source_stem = Path(full_records[0]['source_npy']).stem
pl_pre_p  = mask_root / session / f'{source_stem}_pre.npy'
pl_post_p = mask_root / session / f'{source_stem}_post.npy'
pl_pre  = np.load(pl_pre_p,  mmap_mode='r') if pl_pre_p.exists()  else np.zeros_like(mask_pre)
pl_post = np.load(pl_post_p, mmap_mode='r') if pl_post_p.exists() else np.zeros_like(mask_post)

# Crop pseudolabel to model prob shape (model output already cropped to (H,W) of full_image)
pl_pre  = np.asarray(pl_pre)[: prob_2ch.shape[1], : prob_2ch.shape[2]]
pl_post = np.asarray(pl_post)[: prob_2ch.shape[1], : prob_2ch.shape[2]]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
def _show(ax, arr, cmap, title, vmin=None, vmax=None):
    if vmin is None and vmax is None:
        lo, hi = np.percentile(arr, (1, 99))
        ax.imshow(arr, cmap=cmap, vmin=lo, vmax=max(hi, lo + 1e-3))
    else:
        ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title); ax.axis('off')
_show(axes[0, 0], full_image[0], 'gray', 'PRE input')
_show(axes[0, 1], prob_2ch[0],    'hot', 'pred PRE prob', vmin=0, vmax=1)
_show(axes[0, 2], pl_pre,         'hot', 'pseudo PRE',    vmin=0, vmax=1)
_show(axes[1, 0], full_image[1], 'gray', 'POST input')
_show(axes[1, 1], prob_2ch[1],    'hot', 'pred POST prob', vmin=0, vmax=1)
_show(axes[1, 2], pl_post,        'hot', 'pseudo POST',    vmin=0, vmax=1)
fig.suptitle(f'Full image {demo_idx}: input vs prediction vs pseudo-label', y=1.001)
fig.tight_layout()
fig.savefig(save_dir / 'full_image_inference.png', dpi=140, bbox_inches='tight')
plt.show()


## Synapse mask via PRE-POST co-localisation (post-processing)

In [ ]:
coloc_cfg = SynapseColocCfg(
    prob_thresh       = 0.5,
    min_distance      = 2,
    match_radius_px   = 5,      # ~535 nm at 107 nm/px
    synapse_radius_px = 3,
)

synapse_mask, midpoints, pre_pts, post_pts = build_synapse_mask(
    prob_2ch, coloc_cfg, return_points=True,
)
logger.info(f'pre_pts={len(pre_pts)}  post_pts={len(post_pts)}  '
            f'matched_synapses={len(midpoints)}  '
            f'mask_positive_px={int(synapse_mask.sum())}')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
def _show2(ax, arr, cmap, title):
    lo, hi = np.percentile(arr, (1, 99))
    ax.imshow(arr, cmap=cmap, vmin=lo, vmax=max(hi, lo + 1e-3))
    ax.set_title(title); ax.axis('off')
_show2(axes[0], full_image[0], 'gray', 'PRE input')
_show2(axes[1], full_image[1], 'gray', 'POST input')
axes[2].imshow(full_image[0], cmap='gray',
               vmin=np.percentile(full_image[0], 1),
               vmax=np.percentile(full_image[0], 99))
overlay = np.zeros((*synapse_mask.shape, 4))
overlay[synapse_mask > 0] = [1.0, 1.0, 0.0, 0.6]
axes[2].imshow(overlay)
axes[2].set_title(f'Synapse mask ({int(synapse_mask.sum())} px, {len(midpoints)} sites)')
axes[2].axis('off')
fig.tight_layout()
fig.savefig(save_dir / 'synapse_mask.png', dpi=140, bbox_inches='tight')
plt.show()


## Notes / next steps

- Validation is "Dice against pseudo-labels on a held-out subset" — it is a
  training-progress signal, *not* model selection vs GT. There is no GT.
- If PRE collapses to all-zero or all-one, try
  `JointChannelDiceBCE(bce_weight_pre=0.7, bce_weight_post=1.0)` (plan §6).
- Re-tune `coloc_cfg.match_radius_px` on 10 MIPs by eye before shipping.
- The pseudo-label folder (`data/pseudolabels/spotiflow/`) is static; re-extract
  from `spotiflow.tar.gz` only if the on-disk masks are deleted.
